# Module 7 — A Scikit-learn Transformer

**By the end of this notebook, you will be able to:**
- Explain why any preparation step that *learns* a decision from training data — which columns to drop, which features to keep, a mean, a scaling factor — needs the same guarantee a model already has: fit only on training data, apply everywhere else
- Write a scikit-learn-compatible transformer (`BaseEstimator`, `TransformerMixin`) whose `fit` learns only from the data it is given, and whose `transform` applies that frozen decision to any data it is given afterward
- Chain several transformers — your own and scikit-learn's ready-made ones — into a single `sklearn.pipeline.Pipeline`, evaluated with `cross_val_score`, with no hand-written per-fold loop

**Context:** Two steps you have already built each learn a decision from training data: Module 2's missingness threshold decides which columns to drop, and Module 4's `RFE` decides which features to keep. Both have, until now, been computed once on the whole 4-city dataset, before any `GroupKFold` split ever happens — for each of them, the same question is open: does it matter whether that decision is learned before or after the split? A general way to check, for any such step: recompute its decision after leaving one city out at a time, and see whether it changes. This notebook does not run that check on the real dataset — instead, it builds a transformer whose `fit` is *guaranteed* to only ever see whichever rows it is given, so when you chain several such steps together (a cleaner, a feature selector, a model) in a `Pipeline`, each one's `fit` automatically stays limited to that fold's training rows, for every fold, without you having to check anything yourself.

## Why scikit-learn has a `fit`/`transform` contract

Any real pipeline needs preparation steps that *learn* something from the training data — a mean, a set of columns to drop, a scaling factor — and then *apply* that exact learned decision to other data later, without ever recomputing it. A plain function cannot make that promise: nothing stops you from accidentally calling it again on the wrong data, and — more importantly for this course — tools like [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) and [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) cannot automate "fit only on this fold's training rows, then apply to its validation rows" unless every step in the chain follows the *same* interface.

That interface is two methods and two base classes:
- [`BaseEstimator`](https://scikit-learn.org/stable/modules/generated/sklearn.base.BaseEstimator.html) gives you `get_params()`/`set_params()` for free, which is how `clone()` (used internally by every fold of `cross_val_score`/`GridSearchCV`) can produce a fresh, unfitted copy of your transformer between folds
- [`TransformerMixin`](https://scikit-learn.org/stable/modules/generated/sklearn.base.TransformerMixin.html) gives you `fit_transform(X)` for free, implemented as `fit(X).transform(X)`

The full rulebook is in scikit-learn's own [Developing scikit-learn estimators](https://scikit-learn.org/stable/developers/develop.html) guide. The two rules that matter most here:
- `fit(X, y=None)` learns whatever it needs **from `X` alone**, stores it in attributes ending with `_` (the scikit-learn convention for "set during fit"), and returns `self`
- `transform(X)` applies what was learned to **whatever `X` it is given**, train, validation, or genuinely new data — it never recomputes anything from that `X`

### What scikit-learn's own documentation recommends

Fit only on training data, and use `Pipeline` to make that automatic: this isn't specific to this course. It is stated directly in scikit-learn's own documentation, [*Common pitfalls and recommended practices* — Data Leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage):

> "Data leakage occurs when information that would not be available at prediction time is used when building the model. [...] The general rule is to never call `fit` on the test data. [...] The scikit-learn pipeline is a great way to prevent data leakage as it ensures that the appropriate method is performed on the correct data subset. The pipeline is ideal for use in cross-validation and hyper-parameter tuning functions."

The same page's section on [feature selection during pre-processing](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage-during-pre-processing) — about `SelectKBest`, the same family of tool as Module 4's `RFE` — gives the identical advice: "We recommend using a Pipeline to chain together the feature selection and model estimators." 

In [ ]:
import numpy as np
import pandas as pd

train_toy = pd.DataFrame({"x1": [1.0, 2.0, np.nan, 4.0], "x2": [10.0, np.nan, 30.0, 40.0]})
test_toy = pd.DataFrame({"x1": [np.nan, 5.0], "x2": [100.0, np.nan]})
train_toy

## A plain function cannot keep the promise

Here is the naive way to fill missing values with a column'''s mean:

```python
def fill_with_mean(df):
    return df.fillna(df.mean())
```

The cell below applies it to `train_toy`, then to `test_toy` — the same function, applied the same way.


In [ ]:
def fill_with_mean(df):
    return df.fillna(df.mean())


print("naive on train_toy:")
print(fill_with_mean(train_toy))
print("\nnaive on test_toy:")
print(fill_with_mean(test_toy))

**Look at `test_toy`'s result.** `x1` has one real value (`5.0`) and one `NaN` — `df.mean()` on `test_toy` alone is `5.0`, so the function fills the missing `x1` with `5.0`: it is filling a missing value with *itself*, a circular, meaningless "decision" that only happens because the function recomputes its statistic on whatever it is handed. If `test_toy` represents new data your model will see in production, this is not a training-time convenience going slightly stale — it is silently wrong every single time it runs, because the function was never designed to remember anything from training in the first place. A transformer fixes this structurally: `fit` learns the mean once, from training data only; `transform` reuses that exact number forever after.

## Freeze the decision with a class: `__init__`

`__init__` stores configuration only — every argument, unchanged, as an attribute of the exact same name. Nothing is computed here; that is `fit`'s job. This is not a style preference: `BaseEstimator.get_params()` inspects `__init__`'s signature and reads back attributes of the same name, and `clone()` (which `cross_val_score`/`GridSearchCV` call between every fold) rebuilds a fresh instance from exactly those params. Breaking this rule — computing something in `__init__`, or storing an argument under a different name — makes `clone()` silently produce a different object than you intended.

Our first transformer takes one optional parameter: which columns to impute (`None` means "all of them").

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class ColumnMeanImputer(BaseEstimator, TransformerMixin):
    """Fill missing values with each column's mean, learned once from fit's X."""

    def __init__(self, columns: list[str] | None = None):
        """
        Store which columns to impute; nothing is computed here.

        Parameters:
        - columns: names of the columns to impute; None means every column of X
        """
        self.columns = columns


ColumnMeanImputer().get_params()


## `fit`: learn the means, and only the means

`fit(X, y=None)` receives `X` (and `y`, unused here — it exists only so every transformer accepts the same call signature as an estimator) and must compute the mean of each target column **using only this `X`**. Store the result in `self.means_` (trailing underscore: "set by fit") and return `self`, so `fit` can be chained as `fit(X).transform(X)`. Below, the whole class is written again with `fit` added — redefining a class like this, one method at a time, is a normal thing to do in a notebook.

In [ ]:
# TODO: implement fit(X, y=None): compute the mean of each column in
# self.columns (or every column of X if self.columns is None), store the
# result in self.means_, and return self
class ColumnMeanImputer(BaseEstimator, TransformerMixin):
    """Fill missing values with each column's mean, learned once from fit's X."""
    def __init__(self, columns: list[str] | None = None):
        """
        Store which columns to impute; nothing is computed here.

        Parameters:
        - columns: names of the columns to impute; None means every column of X
        """
    def fit(self, X, y=None):
        """
        Compute the mean of each target column, using only X.

        Parameters:
        - X: DataFrame to learn the column means from
        - y: ignored, present only for scikit-learn's fit(X, y) contract

        Returns:
        - self, so fit can be chained as fit(X).transform(X)
        """


In [ ]:
# Given: try it
imputer = ColumnMeanImputer()
imputer.fit(train_toy)
imputer.means_

## `transform`: apply what was learned, nothing else

`transform(X)` must use `self.means_` — set once, by `fit`, on `train_toy` — to fill missing values in *whatever* `X` it receives now. It never looks at `X`'s own mean.

One detail worth noticing: use `self.means_.index` to know which columns to fill, not `self.columns`. `self.columns` can be `None`; `self.means_.index` is the actual, resolved list of columns `fit` learned means for, whatever `self.columns` was.

In [ ]:
# TODO: implement transform(X): fill missing values in the learned columns
# using self.means_ (never recompute a mean from the X given here); return
# the filled DataFrame
class ColumnMeanImputer(BaseEstimator, TransformerMixin):
    """Fill missing values with each column's mean, learned once from fit's X."""
    def __init__(self, columns: list[str] | None = None):
        """
        Store which columns to impute; nothing is computed here.

        Parameters:
        - columns: names of the columns to impute; None means every column of X
        """
    def fit(self, X, y=None):
        """
        Compute the mean of each target column, using only X.

        Parameters:
        - X: DataFrame to learn the column means from
        - y: ignored, present only for scikit-learn's fit(X, y) contract

        Returns:
        - self, so fit can be chained as fit(X).transform(X)
        """
    def transform(self, X):
        """
        Fill missing values in the learned columns using self.means_.

        Parameters:
        - X: DataFrame to fill — may be different rows than what fit() saw
          (e.g. a validation fold, or genuinely new data)

        Returns:
        - X with missing values in the learned columns filled from self.means_
        """


In [ ]:
# Given: this is a fresh class definition, so fit again, then transform
imputer = ColumnMeanImputer()
imputer.fit(train_toy)
imputer.transform(test_toy)

**Compare to the naive function's result on `test_toy` above.** `x1`'s missing value is now filled with `2.33` — `train_toy`'s mean, learned once and reused — instead of `5.0`, `test_toy`'s own circular mean. Same missing value, same row; the only thing that changed is *where* the fill number came from. That is the entire point of the contract.

## Free extras: `fit_transform` and `clone`

`TransformerMixin` already gave you `fit_transform(X)` — it just calls `fit(X).transform(X)`, nothing more. And `BaseEstimator` gave you compatibility with [`clone()`](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html): given a transformer, `clone()` returns a **new, unfitted** instance built from `get_params()` — the exact mechanism `cross_val_score` and `GridSearchCV` use internally to get a fresh, blank copy for every single fold, so nothing learned on one fold can ever leak into another.

In [ ]:
from sklearn.base import clone

# fit_transform == fit(X).transform(X)
print("fit_transform matches fit().transform():",
      ColumnMeanImputer().fit_transform(train_toy).equals(ColumnMeanImputer().fit(train_toy).transform(train_toy)))

# clone gives a fresh, unfitted copy — same config, no learned state
fresh = clone(imputer)
print("clone keeps params:", fresh.get_params())
print("clone has learned means_:", hasattr(fresh, "means_"))


## Compose it into a real `Pipeline`

The example below chains `ColumnMeanImputer` with a model, with no hand-written per-fold loop this time. That needs a different data shape than `train_toy`/`test_toy`: `GroupKFold`/`cross_val_score` expect one combined dataset (features, a target, and a groups array) that they split themselves, not a train/test pair you already separated by hand. A synthetic one, generated below with scikit-learn'''s own [`make_regression`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_regression.html) — no `air_quality` data needed yet.


In [ ]:
from sklearn.datasets import make_regression

# A synthetic regression dataset (features + target), standing in for
# air_quality: make_regression gives complete data, so a few values are
# blanked out afterward to have something for ColumnMeanImputer to fill,
# and a fake group label per row stands in for "city".
X_arr, y_arr = make_regression(n_samples=40, n_features=3, noise=5.0, random_state=42)
toy_df = pd.DataFrame(X_arr, columns=["f1", "f2", "f3"])
toy_df.loc[[2, 10, 25], "f1"] = np.nan
toy_df.loc[[5, 30], "f2"] = np.nan
groups = np.repeat(["group_a", "group_b", "group_c", "group_d"], 10)
toy_df.assign(group=groups).head(8)


The cell below plugs `ColumnMeanImputer` into a [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) with a model, and evaluates it with [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) + [`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html): they call `fit` **only on each fold's training rows**, then `transform` on both that fold's training and validation rows — automatically, for every fold, because both `ColumnMeanImputer` and the model share the same `fit`/`transform`/`clone` contract.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline

pipe = Pipeline([("imputer", ColumnMeanImputer()), ("model", LinearRegression())])
scores = cross_val_score(
    pipe, toy_df, y_arr, groups=groups, cv=GroupKFold(n_splits=4),
    scoring="neg_root_mean_squared_error",
)
print("rmse per fold:", -scores)
print("rmse mean:", -scores.mean())

**What just happened, fold by fold?** The cell below reproduces fold 0 by hand, to show exactly what `cross_val_score` just did internally.

In [ ]:
from sklearn.metrics import mean_squared_error

# One fold, done by hand: this is what cross_val_score repeats for every fold
train_idx, val_idx = next(iter(GroupKFold(n_splits=4).split(toy_df, groups=groups)))

manual_imputer = ColumnMeanImputer()
manual_imputer.fit(toy_df.iloc[train_idx])  # learns only this fold's training means
manual_model = LinearRegression()
manual_model.fit(manual_imputer.transform(toy_df.iloc[train_idx]), y_arr[train_idx])

# transform() on the held-out rows reuses the means learned above - it never
# looks at the held-out group's own values
manual_preds = manual_model.predict(manual_imputer.transform(toy_df.iloc[val_idx]))
manual_rmse = mean_squared_error(y_arr[val_idx], manual_preds) ** 0.5

print("manual fold 0 rmse:", manual_rmse)
print("cross_val_score's fold 0 rmse:", -scores[0])


## Chain a second transformer

A `Pipeline` is not limited to one preprocessing step. The example below adds [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) — a transformer scikit-learn ships, not one you wrote — right after `ColumnMeanImputer`: fill missing values first, then scale. Nothing about either transformer needs to change for this to work: both already follow the exact same `fit`/`transform` contract, so `Pipeline` chains them for free.


In [ ]:
from sklearn.preprocessing import StandardScaler

chained_pipe = Pipeline([
    ("imputer", ColumnMeanImputer()),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
chained_scores = cross_val_score(
    chained_pipe, toy_df, y_arr, groups=groups, cv=GroupKFold(n_splits=4),
    scoring="neg_root_mean_squared_error",
)
print("rmse per fold:", -chained_scores)
print("rmse mean:", -chained_scores.mean())

**This is the whole idea of chaining:** each step's `fit` still only ever sees that fold's training rows, `transform` still only ever applies what was already learned — `Pipeline` just passes one step's output as the next step's input, for as many steps as you give it. (You'll notice the rmse barely moves: `LinearRegression` is invariant to a linear rescaling of its inputs, so `StandardScaler` does not change what it can learn here — scaling matters far more for models that are sensitive to feature scale. The point of this step is not that the result improved; it's that a second transformer chained in with zero extra effort.) This is exactly how you will combine *two* real preparation steps for `air_quality` next: a cleaner, then a feature selector, then a model.

## Now do it for real: `air_quality`

Everything above was a toy example so you could see the mechanism clearly, with no `air_quality` code involved. Two real preparation steps in this project have the exact same structural gap you just saw generically: Module 2's cleaning (which columns to drop, by missingness) and Module 4's feature selection ([`RFE`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html), which features to keep). Once `AirQualityCleaner` is complete (see Implementation below), you will chain it with `RFE` to fix both at once — its docstring and `tests/test_transformers.py` specify exactly what `fit` and `transform` must do, and `RFE` is already a proper scikit-learn transformer itself, so there is no second class to write for feature selection.

**Continue on the site:** [Module 7 — A Scikit-learn Transformer](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_7/)